# Practice Lab: ViT & CLIP (Lesson 6.4)

Module 6 · 8 exercises
**Needs:** torch, transformers, timm, faiss-cpu

Exercises 1 (patches) and 3 (contrastive loss) run locally.


# Lesson 6.6: Vision Transformers (ViT) & CLIP

8 exercises: 2E + 3M + 3C
**Needs:** `pip install torch torchvision transformers timm faiss-cpu`

Exercises 1 (patch math) and 3 (contrastive loss) run **locally** with PyTorch.


In [ ]:
!pip install torch torchvision transformers timm faiss-cpu matplotlib -q


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math


---
## Exercise 1 (Easy): ViT Patch Visualization

Split 224x224 image into 16x16 patches. Verify dimensions.


In [ ]:
# YOUR CODE
# TODO: create image, unfold into patches
# TODO: print patch count, dimensions, [CLS] token count


<details><summary>Solution</summary>


In [ ]:
import torch

# Synthetic image
img = torch.randn(3, 224, 224)
patch_size = 16

# Method 1: unfold
patches = img.unfold(1, patch_size, patch_size
                    ).unfold(2, patch_size, patch_size)
n_h, n_w = patches.shape[1], patches.shape[2]
n_patches = n_h * n_w

print('ViT Patch Calculation:')
print(f'  Image size: {img.shape[1]}x{img.shape[2]}')
print(f'  Patch size: {patch_size}x{patch_size}')
print(f'  Grid: {n_h}x{n_w} = {n_patches} patches')
print(f'  Each patch pixels: {patch_size}x{patch_size}'
      f' = {patch_size**2}')
print(f'  Flattened (with channels): '
      f'{3}x{patch_size**2} = {3*patch_size**2} dims')
print(f'  With [CLS] token: {n_patches} + 1 = '
      f'{n_patches + 1} tokens')
print()

# Method 2: Conv2d projection (how ViT actually does it)
proj = nn.Conv2d(3, 768, kernel_size=patch_size,
                 stride=patch_size)
projected = proj(img.unsqueeze(0))
# Shape: (1, 768, 14, 14) -> flatten -> (1, 196, 768)
tokens = projected.flatten(2).transpose(1, 2)
print(f'Conv2d projection:')
print(f'  Input: {img.unsqueeze(0).shape}')
print(f'  After conv: {projected.shape}')
print(f'  Tokens: {tokens.shape}')
print(f'  Each token: {tokens.shape[2]}-dim vector')

# Verify different image sizes
for size in [224, 384, 512]:
    n = (size // patch_size) ** 2
    print(f'  {size}x{size} -> {n} patches '
          f'({size//patch_size}x{size//patch_size})')

# Verify: patch_size 14 vs 16
for ps in [14, 16, 32]:
    n = (224 // ps) ** 2
    print(f'  patch_size={ps}: {n} patches '
          f'({224//ps}x{224//ps})')


</details>


---
## Exercise 3 (Medium): Contrastive Loss Implementation

InfoNCE loss from scratch. Verify with perfect vs random alignment.


In [ ]:
# YOUR CODE
# TODO: implement clip_loss
# TODO: test perfect, random, temperature sweep


<details><summary>Solution</summary>


In [ ]:
import torch
import torch.nn.functional as F

def clip_loss(img_emb, txt_emb, temperature=0.07):
    img = F.normalize(img_emb, dim=-1)
    txt = F.normalize(txt_emb, dim=-1)
    logits = (img @ txt.T) / temperature
    labels = torch.arange(len(logits))
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.T, labels)
    return (loss_i2t + loss_t2i) / 2

N, D = 8, 128
print('Contrastive Loss (InfoNCE):')
print(f'  Batch size: {N}, Embedding dim: {D}')
print()

# Test 1: Perfect alignment
shared = F.normalize(torch.randn(N, D), dim=-1)
loss_perfect = clip_loss(shared, shared)
print(f'  Perfect alignment: {loss_perfect:.6f}')
print(f'    (should be ~0.0)')

# Test 2: Random (no alignment)
img_rand = torch.randn(N, D)
txt_rand = torch.randn(N, D)
loss_random = clip_loss(img_rand, txt_rand)
expected_random = math.log(N)
print(f'  Random alignment:  {loss_random:.4f}')
print(f'    (expected ~ln({N}) = {expected_random:.4f})')

# Test 3: Partial alignment (50% correct)
img_partial = torch.randn(N, D)
txt_partial = torch.randn(N, D)
txt_partial[:N//2] = img_partial[:N//2] + 0.1 * torch.randn(N//2, D)
loss_partial = clip_loss(img_partial, txt_partial)
print(f'  Partial alignment: {loss_partial:.4f}')
print(f'    (between perfect and random)')

# Verify ordering
assert loss_perfect < loss_partial < loss_random, 'Order wrong!'
print(f'\n  Ordering verified: perfect < partial < random')

# Temperature sweep
print(f'\n  Temperature sweep (random embeddings):')
for temp in [0.01, 0.07, 0.1, 0.5, 1.0]:
    loss = clip_loss(img_rand, txt_rand, temp)
    print(f'    tau={temp:.2f}: loss={loss:.4f}')

# Batch size effect on difficulty
print(f'\n  Batch size effect (random):')
for n in [4, 8, 16, 32, 64]:
    ie = torch.randn(n, D)
    te = torch.randn(n, D)
    loss = clip_loss(ie, te)
    print(f'    N={n:3d}: loss={loss:.4f} '
          f'(ln({n})={math.log(n):.4f})')
print(f'\n  Key insight: random loss ~ ln(N)')
print(f'  Larger batches = harder task = better learning')


</details>


---
## Exercise 2 (Easy): CLIP Zero-Shot Classifier
Needs GPU/CLIP model. See practice-lab-6_6.html.


In [ ]:
# Exercise 2: CLIP Zero-Shot Classifier
# Needs GPU + transformers.
pass


---
## Exercise 4 (Medium): CLIP Limitation Test
Needs GPU/CLIP model. See practice-lab-6_6.html.


In [ ]:
# Exercise 4: CLIP Limitation Test
# Needs GPU + transformers.
pass


---
## Exercise 5 (Medium): Image Search with FAISS
Needs GPU/CLIP model. See practice-lab-6_6.html.


In [ ]:
# Exercise 5: Image Search with FAISS
# Needs GPU + transformers.
pass


---
## Exercise 6 (Challenge): CLIP Score Evaluator
Needs GPU/CLIP model. See practice-lab-6_6.html.


In [ ]:
# Exercise 6: CLIP Score Evaluator
# Needs GPU + transformers.
pass


---
## Exercise 7 (Challenge): Fine-tune ViT with timm
Needs GPU/CLIP model. See practice-lab-6_6.html.


In [ ]:
# Exercise 7: Fine-tune ViT with timm
# Needs GPU + transformers.
pass


---
## Exercise 8 (Challenge): Indian E-Commerce Visual Search
Needs GPU/CLIP model. See practice-lab-6_6.html.


In [ ]:
# Exercise 8: Indian E-Commerce Visual Search
# Needs GPU + transformers.
pass
